In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder \
    .appName("Lab2_Full_ETL_Postgres") \
    .config("spark.jars", "/home/jovyan/jars/postgresql-42.7.2.jar,/home/jovyan/jars/clickhouse-jdbc-0.6.0-all.jar") \
    .config("spark.driver.extraClassPath", "/home/jovyan/jars/postgresql-42.7.2.jar:/home/jovyan/jars/clickhouse-jdbc-0.6.0-all.jar") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

pgUrl = "jdbc:postgresql://postgres:5432/postgres"
pgProperties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

In [3]:
rawData = spark.read.jdbc(url=pgUrl, table="raw_data", properties=pgProperties)

def writeTable(df, tableName):
    df.write.jdbc(url=pgUrl, table=tableName, mode="overwrite", properties=pgProperties)
    print(f"Table {tableName} was written successful")

In [4]:
dimGeography = rawData.select(
    F.col("customer_country").alias("country"),
    F.lit(None).cast("string").alias("city"),
    F.lit(None).cast("string").alias("state"),
    F.col("customer_postal_code").alias("postalCode")
).union(
    rawData.select(
        F.col("store_country").alias("country"), 
        F.col("store_city").alias("city"), 
        F.col("store_state").alias("state"), 
        F.lit(None).cast("string").alias("postalCode")
    )
).union(
    rawData.select(
        F.col("supplier_country").alias("country"), 
        F.col("supplier_city").alias("city"), 
        F.lit(None).cast("string").alias("state"), 
        F.lit(None).cast("string").alias("postalCode")
    )
).distinct().dropna(subset=["country"]) \
.withColumn("geoId", F.monotonically_increasing_id())

writeTable(dimGeography, "DimGeography")

Table DimGeography was written successful


In [5]:
dimProductCategories = rawData.select(
    F.col("product_category").alias("categoryName"),
    F.col("pet_category").alias("petCategory")
).distinct().dropna() \
.withColumn("categoryId", F.monotonically_increasing_id())

writeTable(dimProductCategories, "DimProductCategories")

Table DimProductCategories was written successful


In [6]:
dimCustomers = rawData.select(
    F.col("customer_first_name").alias("firstName"),
    F.col("customer_last_name").alias("lastName"),
    F.col("customer_age").alias("age"),
    F.col("customer_email").alias("email"),
    F.col("customer_country").alias("country"),
    F.col("customer_postal_code").alias("postalCode"),
    F.col("customer_pet_type").alias("petType"),
    F.col("customer_pet_name").alias("petName"),
    F.col("customer_pet_breed").alias("petBreed")
).dropDuplicates(["email"]) \
.join(dimGeography, ["country", "postalCode"], "left") \
.select("firstName", "lastName", "age", "email", "geoId", "petType", "petName", "petBreed") \
.withColumn("customerId", F.monotonically_increasing_id())

writeTable(dimCustomers, "DimCustomers")

Table DimCustomers was written successful


In [7]:
dimSellers = rawData.select(
    F.col("seller_first_name").alias("firstName"),
    F.col("seller_last_name").alias("lastName"),
    F.col("seller_email").alias("email"),
    F.col("seller_country").alias("country"),
    F.col("seller_postal_code").alias("postalCode")
).dropDuplicates(["email"]) \
.join(dimGeography, ["country", "postalCode"], "left") \
.select("firstName", "lastName", "email", "geoId") \
.withColumn("sellerId", F.monotonically_increasing_id())

writeTable(dimSellers, "DimSellers")

Table DimSellers was written successful


In [8]:
dimSuppliers = rawData.select(
    F.col("supplier_name").alias("name"),
    F.col("supplier_contact").alias("contactName"),
    F.col("supplier_email").alias("email"),
    F.col("supplier_phone").alias("phone"),
    F.col("supplier_country").alias("country"),
    F.col("supplier_city").alias("city")
).dropDuplicates(["name", "email"]) \
.join(dimGeography, ["country", "city"], "left") \
.select("name", "contactName", "email", "phone", "geoId") \
.withColumn("supplierId", F.monotonically_increasing_id())

writeTable(dimSuppliers, "DimSuppliers")

Table DimSuppliers was written successful


In [9]:
dimStores = rawData.select(
    F.col("store_name").alias("name"),
    F.col("store_location").alias("locationAddress"),
    F.col("store_phone").alias("phone"),
    F.col("store_email").alias("email"),
    F.col("store_country").alias("country"),
    F.col("store_city").alias("city"),
    F.col("store_state").alias("state")
).dropDuplicates(["name"]) \
.join(dimGeography, ["country", "city", "state"], "left") \
.select("name", "locationAddress", "phone", "email", "geoId") \
.withColumn("storeId", F.monotonically_increasing_id())

writeTable(dimStores, "DimStores")

Table DimStores was written successful


In [10]:
dimProducts = rawData.select(
    F.col("product_name").alias("name"),
    F.col("product_category").alias("categoryName"),
    F.col("pet_category").alias("petCategory"),
    F.col("product_brand").alias("brand"),
    F.col("product_price").alias("price"),
    F.col("product_weight").alias("weight"),
    F.col("product_color").alias("color"),
    F.col("product_size").alias("size"),
    F.col("product_material").alias("material"),
    F.col("product_description").alias("description"),
    F.col("product_rating").alias("rating"),
    F.col("product_reviews").alias("reviews"),
    F.col("product_release_date").alias("releaseDate"),
    F.col("product_expiry_date").alias("expiryDate")
).dropDuplicates(["name", "brand"]) \
.join(dimProductCategories, ["categoryName", "petCategory"], "left") \
.select("name", "categoryId", "brand", "price", "weight", "color", "size", "material", "description", "rating", "reviews", "releaseDate", "expiryDate") \
.withColumn("productId", F.monotonically_increasing_id())

writeTable(dimProducts, "DimProducts")

Table DimProducts was written successful


In [11]:
dimCustomersClean = dimCustomers.dropDuplicates(["email"])
dimSellersClean = dimSellers.dropDuplicates(["email"])
dimProductsClean = dimProducts.dropDuplicates(["name", "brand"])
dimStoresClean = dimStores.dropDuplicates(["name"])
dimSuppliersClean = dimSuppliers.dropDuplicates(["name", "email"])

factSales = rawData.alias("r") \
    .join(dimCustomersClean.alias("c"), F.col("r.customer_email") == F.col("c.email"), "left") \
    .join(dimSellersClean.alias("sel"), F.col("r.seller_email") == F.col("sel.email"), "left") \
    .join(dimProductsClean.alias("p"), (F.col("r.product_name") == F.col("p.name")) & (F.col("r.product_brand") == F.col("p.brand")), "left") \
    .join(dimStoresClean.alias("st"), F.col("r.store_name") == F.col("st.name"), "left") \
    .join(dimSuppliersClean.alias("sup"), (F.col("r.supplier_name") == F.col("sup.name")) & (F.col("r.supplier_email") == F.col("sup.email")), "left") \
    .select(
        F.monotonically_increasing_id().alias("saleId"),
        F.col("r.sale_date").alias("saleDate"),
        F.col("c.customerId"),
        F.col("sel.sellerId"),
        F.col("p.productId"),
        F.col("st.storeId"),
        F.col("sup.supplierId"),
        F.col("r.sale_quantity").alias("quantity"),
        F.col("r.sale_total_price").alias("totalPrice")
    )

writeTable(factSales, "FactSales")

Table FactSales was written successful
